# RAG Math and Retrieval — Exercises

[← Quantization and Distillation](../11-Quantization-and-Distillation/notes.md) | [Home](../../README.md) | [Serving and Systems Tradeoffs →](../13-Serving-and-Systems-Tradeoffs/notes.md)

8 exercises covering BM25 scoring, contrastive loss, HNSW memory, HyDE pipeline,
RRF fusion, chunking analysis, NDCG calculation, and RAG marginalisation.

In [ ]:
# === Setup ===
import numpy as np
np.random.seed(42)

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

def print_table(headers, rows, col_width=14):
    hdr = ' | '.join(h.center(col_width) for h in headers)
    print(hdr)
    print('-' * len(hdr))
    for row in rows:
        print(' | '.join(str(v).center(col_width) for v in row))

def fmt_sci(x, p=4):
    return f'{x:.{p}e}'

def fmt_vec(v, p=4):
    return '[' + ', '.join(f'{x:.{p}f}' for x in v) + ']'

def fmt_bytes(n_bytes):
    if n_bytes >= 1e9:
        return f'{n_bytes/1e9:.2f} GB'
    elif n_bytes >= 1e6:
        return f'{n_bytes/1e6:.2f} MB'
    return f'{n_bytes/1e3:.2f} KB'

def softmax(x, temperature=1.0):
    x = np.asarray(x, dtype=np.float64)
    z = x / temperature
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()

def kl_divergence(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    mask = p > 1e-15
    return np.sum(p[mask] * np.log(p[mask] / q[mask]))

def entropy(p):
    p = np.asarray(p, dtype=np.float64)
    mask = p > 1e-15
    return -np.sum(p[mask] * np.log(p[mask]))

print('Setup complete ✓')

## Exercise 1: BM25 by Hand

Given 3 documents and query "neural retrieval":

- D0: "neural networks learn representations from data"
- D1: "information retrieval finds relevant documents given a query"
- D2: "neural retrieval models use dense embeddings for search"

Compute with k₁ = 1.5, b = 0.75:

1. Tokenize each document (whitespace split, lowercase)
2. Compute raw term frequency for "neural" and "retrieval" in each doc
3. Compute document lengths and average document length
4. Compute IDF for each query term: log((N - df + 0.5) / (df + 0.5))
5. Compute BM25 score for each document
6. Rank the documents

In [ ]:
# === Exercise 1: Scaffold ===

docs = [
    "neural networks learn representations from data",
    "information retrieval finds relevant documents given a query",
    "neural retrieval models use dense embeddings for search",
]
query = "neural retrieval"
k1, b = 1.5, 0.75
N = len(docs)

# TODO: Step 1 — Tokenize
doc_tokens = None  # list of lists of tokens
query_tokens = None  # list of query tokens

# TODO: Step 2 — Term frequency for each query term in each doc
# tf[doc_idx][term] = count

# TODO: Step 3 — Document lengths and avgdl
# dl = [len of each doc_tokens]
# avgdl = mean of dl

# TODO: Step 4 — IDF for "neural" and "retrieval"
# df(t) = number of docs containing term t
# idf(t) = log((N - df + 0.5) / (df + 0.5))

# TODO: Step 5 — BM25 score for each doc
# For each query term t in doc d:
#   score += idf(t) * (f * (k1+1)) / (f + k1*(1 - b + b*dl/avgdl))

# TODO: Step 6 — Rank and print results
print("Your BM25 scores and ranking:")

In [ ]:
# === Exercise 1: Solution ===

docs = [
    "neural networks learn representations from data",
    "information retrieval finds relevant documents given a query",
    "neural retrieval models use dense embeddings for search",
]
query = "neural retrieval"
k1, b = 1.5, 0.75
N = len(docs)

# Step 1: Tokenize
doc_tokens = [d.lower().split() for d in docs]
query_tokens = query.lower().split()
print("Tokens:")
for i, tokens in enumerate(doc_tokens):
    print(f"  D{i}: {tokens}")
print(f"  Query: {query_tokens}")

# Step 2: Term frequency
print("\nTerm frequencies:")
for i, tokens in enumerate(doc_tokens):
    tf = {}
    for t in tokens:
        tf[t] = tf.get(t, 0) + 1
    for qt in query_tokens:
        print(f"  D{i}: tf('{qt}') = {tf.get(qt, 0)}")

# Step 3: Document lengths
dl = [len(t) for t in doc_tokens]
avgdl = np.mean(dl)
print(f"\nDocument lengths: {dl}")
print(f"Average doc length: {avgdl:.2f}")

# Step 4: IDF
print("\nIDF computation:")
for qt in query_tokens:
    df = sum(1 for tokens in doc_tokens if qt in tokens)
    idf = np.log((N - df + 0.5) / (df + 0.5))
    print(f"  '{qt}': df={df}, IDF = log(({N}-{df}+0.5)/({df}+0.5)) = {idf:.4f}")

# Step 5: BM25 scores
print("\nBM25 scores:")
scores = []
for i, tokens in enumerate(doc_tokens):
    tf = {}
    for t in tokens:
        tf[t] = tf.get(t, 0) + 1
    
    score = 0.0
    details = []
    for qt in query_tokens:
        f = tf.get(qt, 0)
        df = sum(1 for d in doc_tokens if qt in d)
        idf = np.log((N - df + 0.5) / (df + 0.5))
        
        if f > 0:
            tf_component = (f * (k1 + 1)) / (f + k1 * (1 - b + b * dl[i] / avgdl))
            term_score = idf * tf_component
            details.append(f"  '{qt}': IDF={idf:.4f} × TF_component={tf_component:.4f} = {term_score:.4f}")
        else:
            term_score = 0.0
            details.append(f"  '{qt}': not in doc → 0.0000")
        score += term_score
    
    scores.append(score)
    print(f"\n  D{i} (dl={dl[i]}):")
    for detail in details:
        print(f"    {detail}")
    print(f"    BM25(q, D{i}) = {score:.4f}")

# Step 6: Ranking
ranking = np.argsort(scores)[::-1]
print(f"\nFinal Ranking:")
for rank, idx in enumerate(ranking):
    print(f"  Rank {rank+1}: D{idx} (score: {scores[idx]:.4f})")
print(f"\nD2 ranks highest — contains both 'neural' and 'retrieval'")

## Exercise 2: Contrastive Loss at Different Temperatures

Given a batch of 4 (query, positive) pairs with computed similarity matrix:

```
        P0     P1     P2     P3
Q0    0.85   0.15  -0.10   0.20
Q1    0.10   0.78   0.05   0.12
Q2   -0.05   0.08   0.90   0.03
Q3    0.20   0.15   0.10   0.82
```

Diagonal = positive pairs; off-diagonal = in-batch negatives.

1. Compute InfoNCE loss at τ = 0.1
2. Compute InfoNCE loss at τ = 1.0
3. Compute per-query softmax probabilities at both temperatures
4. Interpret: which temperature gives sharper discrimination?

In [ ]:
# === Exercise 2: Scaffold ===

sim_matrix = np.array([
    [0.85, 0.15, -0.10, 0.20],
    [0.10, 0.78,  0.05, 0.12],
    [-0.05, 0.08, 0.90, 0.03],
    [0.20, 0.15,  0.10, 0.82],
])
B = 4

# TODO: Step 1 — InfoNCE at τ = 0.1
# Scale: sim / τ
# Stabilise: subtract max per row
# Loss = -mean(log(exp(diag) / sum(exp(row))))
tau_01 = 0.1
# loss_01 = ?

# TODO: Step 2 — InfoNCE at τ = 1.0
tau_10 = 1.0
# loss_10 = ?

# TODO: Step 3 — Softmax probabilities per query at both τ
# probs[i] = softmax(sim_matrix[i] / τ)

# TODO: Step 4 — Interpret temperature effect
print("Your InfoNCE losses and interpretation:")

In [ ]:
# === Exercise 2: Solution ===

sim_matrix = np.array([
    [0.85, 0.15, -0.10, 0.20],
    [0.10, 0.78,  0.05, 0.12],
    [-0.05, 0.08, 0.90, 0.03],
    [0.20, 0.15,  0.10, 0.82],
])
B = 4

def infonce_loss(sim, tau):
    """InfoNCE contrastive loss."""
    scaled = sim / tau
    scaled = scaled - np.max(scaled, axis=1, keepdims=True)  # stability
    exp_sim = np.exp(scaled)
    log_probs = scaled[np.arange(B), np.arange(B)] - np.log(np.sum(exp_sim, axis=1))
    return -np.mean(log_probs)

# Step 1: τ = 0.1
loss_01 = infonce_loss(sim_matrix, 0.1)
print(f"InfoNCE loss at τ=0.1: {loss_01:.4f}")

# Step 2: τ = 1.0
loss_10 = infonce_loss(sim_matrix, 1.0)
print(f"InfoNCE loss at τ=1.0: {loss_10:.4f}")

# Step 3: Softmax probabilities
print(f"\n--- Softmax Probabilities ---")
for tau_val, tau_label in [(0.1, 'τ=0.1'), (1.0, 'τ=1.0')]:
    print(f"\n  {tau_label}:")
    for i in range(B):
        probs = softmax(sim_matrix[i], temperature=tau_val)
        pos_prob = probs[i]
        max_neg = max(probs[j] for j in range(B) if j != i)
        print(f"    Q{i}: P(positive) = {pos_prob:.6f}, "
              f"max P(negative) = {max_neg:.6f}, "
              f"ratio = {pos_prob/max(max_neg, 1e-15):.1f}×")

# Step 4: Interpretation
print(f"\n--- Interpretation ---")
print(f"τ=0.1: loss={loss_01:.4f} — very sharp; positive dominates exponentially")
print(f"τ=1.0: loss={loss_10:.4f} — softer; harder for model to distinguish")
print(f"Lower τ → lower loss (easier to separate)")
print(f"But very low τ can cause gradient issues (vanishing for easy examples)")
print(f"\nTraining typically starts τ=0.07–0.2 and may be learnable")

# Accuracy at different temperatures
print(f"\n--- Top-1 Accuracy ---")
for tau in [0.01, 0.05, 0.1, 0.5, 1.0, 5.0]:
    scaled = sim_matrix / tau
    preds = np.argmax(scaled, axis=1)
    acc = np.mean(preds == np.arange(B))
    print(f"  τ={tau:.2f}: accuracy = {acc:.0%}")

## Exercise 3: HNSW Parameter Tuning

For a corpus of 1M vectors with d = 768:

1. Estimate memory for M = 16 vs M = 32
   - Graph memory: N × avg_levels × M × 4 bytes (avg_levels ≈ 1 + 1/ln(M))
   - Vector memory: N × d × 2 bytes (BF16)
2. Estimate theoretical recall@10 improvement from M = 16 → 32
3. Estimate query latency given ef_search = 200 and 1μs per distance computation
4. Recommend configuration for latency < 5ms at 95%+ recall

In [ ]:
# === Exercise 3: Scaffold ===

N = 1_000_000  # corpus size
d = 768        # embedding dimension

# TODO: Step 1 — Memory estimation
# For M in [16, 32]:
#   avg_levels = 1 + 1/ln(M)
#   graph_mem = N * avg_levels * M * 4  (bytes per neighbour ID)
#   vec_mem = N * d * 2  (BF16)
#   total = graph_mem + vec_mem

# TODO: Step 2 — Recall estimation
# Higher M → more connections → better recall
# Rule of thumb: M=16 → ~95% recall@10; M=32 → ~98% recall@10

# TODO: Step 3 — Latency estimation
# ef_search = 200 candidates examined
# Each: 1 distance computation (d multiplies) ≈ 1μs
# Plus graph traversal overhead

# TODO: Step 4 — Recommendation
print("Your HNSW configuration analysis:")

In [ ]:
# === Exercise 3: Solution ===

N = 1_000_000
d = 768

print("=" * 60)
print("HNSW Parameter Analysis for 1M × 768d")
print("=" * 60)

# Step 1: Memory estimation
print("\n--- Step 1: Memory Estimation ---")
headers = ['Parameter', 'M=16', 'M=32', 'M=64']
rows = []

for M in [16, 32, 64]:
    avg_levels = 1 + 1 / np.log(M)
    graph_mem = N * avg_levels * M * 4  # 4 bytes per neighbour ID
    vec_mem_bf16 = N * d * 2            # BF16 vectors
    vec_mem_fp32 = N * d * 4            # FP32 vectors
    total_bf16 = graph_mem + vec_mem_bf16
    total_fp32 = graph_mem + vec_mem_fp32
    
    if M == 16:
        col = 'M=16'
    elif M == 32:
        col = 'M=32'
    else:
        col = 'M=64'
    
    print(f"\n  {col}:")
    print(f"    avg_levels = 1 + 1/ln({M}) = {avg_levels:.3f}")
    print(f"    graph memory = {N} × {avg_levels:.3f} × {M} × 4 = {fmt_bytes(graph_mem)}")
    print(f"    vector memory (BF16) = {N} × {d} × 2 = {fmt_bytes(vec_mem_bf16)}")
    print(f"    total (BF16) = {fmt_bytes(total_bf16)}")
    print(f"    total (FP32) = {fmt_bytes(total_fp32)}")

# Step 2: Recall estimation
print("\n--- Step 2: Recall Estimation ---")
print("Empirical recall@10 ranges (ef_search=200):")
configs = [
    (16, "93–96%", "Standard; good balance"),
    (32, "97–99%", "Higher recall; 2× graph memory"),
    (64, "99%+",   "Diminishing returns; 4× graph memory"),
]
for M, recall_range, note in configs:
    print(f"  M={M:2d}: recall@10 ≈ {recall_range}  ({note})")

# Step 3: Latency estimation
print("\n--- Step 3: Query Latency Estimation ---")
ef_search = 200
time_per_dist_us = 1.0  # microseconds per d=768 dot product

for M in [16, 32]:
    avg_levels = 1 + 1 / np.log(M)
    # Greedy descent: ~log(N) steps through upper layers
    upper_steps = int(np.log2(N) * avg_levels)
    # Bottom layer: examine ef_search candidates
    bottom_comparisons = ef_search * M  # each candidate checks M neighbours
    
    total_comparisons = upper_steps + bottom_comparisons
    latency_us = total_comparisons * time_per_dist_us
    latency_ms = latency_us / 1000
    
    print(f"  M={M:2d}, ef_search={ef_search}:")
    print(f"    upper layer steps: ~{upper_steps}")
    print(f"    bottom comparisons: ~{bottom_comparisons}")
    print(f"    total: ~{total_comparisons} distance computations")
    print(f"    estimated latency: {latency_ms:.2f} ms")

# Step 4: Recommendation
print("\n--- Step 4: Recommendation (latency < 5ms, recall > 95%) ---")
print("  M = 16, ef_search = 200:")
M_rec = 16
avg_levels = 1 + 1 / np.log(M_rec)
graph_mem = N * avg_levels * M_rec * 4
vec_mem = N * d * 2
total_mem = graph_mem + vec_mem
est_comparisons = int(np.log2(N) * avg_levels) + 200 * M_rec
est_latency_ms = est_comparisons * time_per_dist_us / 1000

print(f"    Memory:  {fmt_bytes(total_mem)}")
print(f"    Latency: ~{est_latency_ms:.2f} ms (< 5ms ✓)")
print(f"    Recall:  ~95% ✓")
print(f"    Verdict: M=16 sufficient; M=32 only if recall > 98% required")
print(f"\n  If recall critical, combine HNSW (M=16) + flat reranking of top-100")

## Exercise 4: HyDE Pipeline

Query: "What causes transformer training instability?"

1. Write a prompt that asks an LLM to generate a hypothetical answer (no retrieval)
2. Explain why embedding the hypothetical answer helps retrieval
3. Simulate: generate a query vector and a hypothetical doc vector
4. Show retrieval improvement with HyDE vs direct query
5. Discuss failure cases — when does HyDE hurt?

In [ ]:
# === Exercise 4: Scaffold ===

query = "What causes transformer training instability?"

# TODO: Step 1 — Write a HyDE prompt
# The prompt should instruct the LLM to generate a paragraph
# answering the query as if it were a passage from a textbook.
# hyde_prompt = "..."

# TODO: Step 2 — Explain the distribution gap
# Queries are short questions → different embedding distribution
# Documents are long passages → different embedding distribution
# HyDE bridges this gap by transforming query → document space

# TODO: Step 3 — Simulate embeddings
# query_vec: high variance, weakly aligned with topic
# hyde_vec: low variance, well-aligned with document cluster

# TODO: Step 4 — Compare retrieval precision

# TODO: Step 5 — Discuss failure cases
print("Your HyDE analysis:")

In [ ]:
# === Exercise 4: Solution ===

query = "What causes transformer training instability?"

# Step 1: HyDE prompt
print("--- Step 1: HyDE Prompt ---")
hyde_prompt = """Write a short, factual paragraph that answers the following question 
as if it were an excerpt from a machine learning textbook. Do not mention that 
you are writing an answer — just write the passage directly.

Question: What causes transformer training instability?

Passage:"""

print(hyde_prompt)

hypothetical_answer = """Transformer training instability is primarily caused by 
exploding or vanishing gradients in deep attention layers, particularly when 
learning rate is too high or when layer normalisation is improperly configured. 
The attention mechanism's softmax operation can produce very peaked distributions 
early in training, leading to large gradient magnitudes. Additionally, the 
residual stream can accumulate activations that grow with depth, causing loss 
spikes. Common mitigations include learning rate warmup, gradient clipping, 
pre-layer normalisation (Pre-LN), and careful initialisation of attention weights 
with small scaling factors."""

print(f"\n--- Hypothetical Answer ---")
print(hypothetical_answer.strip())

# Step 2: Distribution gap explanation
print(f"\n--- Step 2: Why HyDE Works ---")
print("Query:  short question (7 tokens), interrogative, high-level")
print("Passage: long answer (80+ tokens), declarative, detailed")
print("These live in DIFFERENT regions of embedding space.")
print("HyDE transforms query → hypothetical passage → same distribution as docs.")
print("Embedding of hypothetical answer is closer to relevant passages than")
print("embedding of the original question.")

# Step 3: Simulated retrieval
np.random.seed(42)
d = 64

# Document corpus about ML topics
topic_centroids = {
    'training_stability': np.random.randn(d) * 2,
    'optimisation': np.random.randn(d) * 2,
    'attention': np.random.randn(d) * 2,
    'nlp_tasks': np.random.randn(d) * 2,
    'computer_vision': np.random.randn(d) * 2,
}

# Generate 500 docs
n_docs = 500
docs_per_topic = 100
doc_vecs = []
doc_labels = []
for topic_name, centroid in topic_centroids.items():
    vecs = centroid + 0.4 * np.random.randn(docs_per_topic, d)
    doc_vecs.append(vecs)
    doc_labels.extend([topic_name] * docs_per_topic)
doc_vecs = np.vstack(doc_vecs)
doc_vecs /= np.linalg.norm(doc_vecs, axis=1, keepdims=True)

# Query embedding: weakly aligned (short question → noisy)
query_vec = topic_centroids['training_stability'] * 0.2 + np.random.randn(d) * 1.5
query_vec /= np.linalg.norm(query_vec)

# HyDE embedding: well-aligned (hypothetical passage → close to docs)
hyde_vec = topic_centroids['training_stability'] + 0.3 * np.random.randn(d)
hyde_vec /= np.linalg.norm(hyde_vec)

# Step 4: Compare retrieval
print(f"\n--- Step 4: Retrieval Comparison ---")
target_topic = 'training_stability'

# Direct query retrieval
query_scores = doc_vecs @ query_vec
query_top10 = np.argsort(query_scores)[::-1][:10]
query_precision = np.mean([doc_labels[i] == target_topic for i in query_top10])

# HyDE retrieval
hyde_scores = doc_vecs @ hyde_vec
hyde_top10 = np.argsort(hyde_scores)[::-1][:10]
hyde_precision = np.mean([doc_labels[i] == target_topic for i in hyde_top10])

print(f"Direct query — Precision@10: {query_precision:.0%}")
print(f"HyDE query  — Precision@10: {hyde_precision:.0%}")
print(f"Improvement: {(hyde_precision - query_precision)*100:+.0f} percentage points")

# Cosine to target centroid
target_centroid_norm = topic_centroids['training_stability'] / np.linalg.norm(topic_centroids['training_stability'])
print(f"\nCosine to target topic centroid:")
print(f"  Direct query: {query_vec @ target_centroid_norm:.4f}")
print(f"  HyDE vector:  {hyde_vec @ target_centroid_norm:.4f}")

# Step 5: Failure cases
print(f"\n--- Step 5: When HyDE Hurts ---")
print("1. LLM generates WRONG hypothetical → embeddings mislead retrieval")
print("2. Ambiguous queries → LLM guesses wrong interpretation")
print("3. Factoid queries ('When was BERT published?') → query already specific")
print("4. Latency-sensitive: adds LLM inference before retrieval (~200ms+)")
print("5. Topics outside LLM knowledge → hallucinated hypothesis → wrong cluster")

# Demonstrate failure: wrong hypothesis
wrong_hyde = topic_centroids['computer_vision'] + 0.3 * np.random.randn(d)
wrong_hyde /= np.linalg.norm(wrong_hyde)
wrong_scores = doc_vecs @ wrong_hyde
wrong_top10 = np.argsort(wrong_scores)[::-1][:10]
wrong_precision = np.mean([doc_labels[i] == target_topic for i in wrong_top10])
print(f"\nWrong HyDE (CV topic instead) — Precision@10: {wrong_precision:.0%}")
print("Bad hypothesis → worse than direct query!")

## Exercise 5: RRF Fusion

BM25 ranks: Doc A at position 3, Doc B at position 7
Dense ranks: Doc A at position 8, Doc B at position 2

1. Compute RRF score for Doc A with k = 60
2. Compute RRF score for Doc B with k = 60
3. Which document ranks higher after fusion?
4. Explore: how does k affect the relative ranking?
5. Add a third system (ColBERT) and recompute

In [ ]:
# === Exercise 5: Scaffold ===

k = 60  # RRF smoothing constant

# BM25 ranks (1-indexed)
bm25_rank_A = 3
bm25_rank_B = 7

# Dense ranks (1-indexed)
dense_rank_A = 8
dense_rank_B = 2

# TODO: Step 1 — RRF(A) = 1/(k + rank_bm25(A)) + 1/(k + rank_dense(A))
# rrf_A = ?

# TODO: Step 2 — RRF(B) = 1/(k + rank_bm25(B)) + 1/(k + rank_dense(B))
# rrf_B = ?

# TODO: Step 3 — Compare and rank

# TODO: Step 4 — Vary k and see effect on ranking

# TODO: Step 5 — Add ColBERT ranks and recompute
print("Your RRF analysis:")

In [ ]:
# === Exercise 5: Solution ===

k = 60

bm25_rank_A, bm25_rank_B = 3, 7
dense_rank_A, dense_rank_B = 8, 2

# Step 1: RRF(A)
rrf_A = 1/(k + bm25_rank_A) + 1/(k + dense_rank_A)
print(f"--- Step 1: RRF(A) ---")
print(f"  1/(60+3) + 1/(60+8) = 1/63 + 1/68")
print(f"  = {1/63:.6f} + {1/68:.6f}")
print(f"  = {rrf_A:.6f}")

# Step 2: RRF(B)
rrf_B = 1/(k + bm25_rank_B) + 1/(k + dense_rank_B)
print(f"\n--- Step 2: RRF(B) ---")
print(f"  1/(60+7) + 1/(60+2) = 1/67 + 1/62")
print(f"  = {1/67:.6f} + {1/62:.6f}")
print(f"  = {rrf_B:.6f}")

# Step 3: Compare
print(f"\n--- Step 3: Final Ranking ---")
print(f"  RRF(A) = {rrf_A:.6f}")
print(f"  RRF(B) = {rrf_B:.6f}")
winner = 'B' if rrf_B > rrf_A else 'A'
print(f"  Winner: Doc {winner}")
print(f"  Doc B wins because rank 2 in dense (1/62) > Doc A's rank 3 in BM25 (1/63)")

# Step 4: Effect of k
print(f"\n--- Step 4: Effect of k ---")
headers = ['k', 'RRF(A)', 'RRF(B)', 'Winner', 'Margin']
rows = []
for k_val in [1, 5, 10, 30, 60, 100, 500]:
    rA = 1/(k_val + bm25_rank_A) + 1/(k_val + dense_rank_A)
    rB = 1/(k_val + bm25_rank_B) + 1/(k_val + dense_rank_B)
    w = 'A' if rA > rB else 'B'
    margin = abs(rA - rB) / max(rA, rB) * 100
    rows.append([k_val, f'{rA:.6f}', f'{rB:.6f}', f'Doc {w}', f'{margin:.2f}%'])
print_table(headers, rows, col_width=12)
print("\nSmaller k → top ranks weighted more heavily → Doc B's rank-2 matters more")
print("Larger k → more uniform → margin shrinks")

# Step 5: Three-way fusion
print(f"\n--- Step 5: Three-Way with ColBERT ---")
colbert_rank_A = 5
colbert_rank_B = 4

k = 60
rrf_A_3 = 1/(k+bm25_rank_A) + 1/(k+dense_rank_A) + 1/(k+colbert_rank_A)
rrf_B_3 = 1/(k+bm25_rank_B) + 1/(k+dense_rank_B) + 1/(k+colbert_rank_B)

print(f"ColBERT ranks: A at {colbert_rank_A}, B at {colbert_rank_B}")
print(f"RRF_3(A) = 1/63 + 1/68 + 1/65 = {rrf_A_3:.6f}")
print(f"RRF_3(B) = 1/67 + 1/62 + 1/64 = {rrf_B_3:.6f}")
winner_3 = 'B' if rrf_B_3 > rrf_A_3 else 'A'
print(f"Winner: Doc {winner_3}")
print(f"\nRRF is additive — each system contributes independently")
print(f"No score normalisation needed — only ranks matter")

## Exercise 6: Chunking Analysis

Given a 2000-token document:

**Strategy A:** Fixed-size chunking, 256 tokens, 64-token overlap
**Strategy B:** Semantic chunking, average 180 tokens, 0 overlap

1. Compute number of chunks for each strategy
2. Compute total tokens stored (including overlap)
3. Compute storage overhead vs original document
4. Compare retrieval precision tradeoffs
5. Discuss: which is better for different content types?

In [ ]:
# === Exercise 6: Scaffold ===

doc_length = 2000  # tokens

# Strategy A: Fixed-size
chunk_size_A = 256
overlap_A = 64

# Strategy B: Semantic
avg_chunk_size_B = 180
overlap_B = 0

# TODO: Step 1 — Number of chunks
# Strategy A: stride = chunk_size - overlap = 192
# n_chunks_A = ceil((doc_length - overlap) / stride)
# Strategy B: n_chunks_B = ceil(doc_length / avg_chunk_size)

# TODO: Step 2 — Total tokens stored (accounting for overlap)

# TODO: Step 3 — Storage overhead

# TODO: Step 4 — Retrieval precision tradeoffs

# TODO: Step 5 — Content type discussion
print("Your chunking analysis:")

In [ ]:
# === Exercise 6: Solution ===

doc_length = 2000

# Strategy A: Fixed-size
chunk_size_A = 256
overlap_A = 64
stride_A = chunk_size_A - overlap_A  # 192

# Strategy B: Semantic
avg_chunk_size_B = 180
overlap_B = 0

# Step 1: Number of chunks
n_chunks_A = int(np.ceil((doc_length - overlap_A) / stride_A))
n_chunks_B = int(np.ceil(doc_length / avg_chunk_size_B))

print("--- Step 1: Number of Chunks ---")
print(f"Strategy A (fixed 256, overlap 64):")
print(f"  Stride = {chunk_size_A} - {overlap_A} = {stride_A}")
print(f"  Chunks = ceil(({doc_length} - {overlap_A}) / {stride_A}) = {n_chunks_A}")
print(f"\nStrategy B (semantic, avg 180):")
print(f"  Chunks = ceil({doc_length} / {avg_chunk_size_B}) = {n_chunks_B}")

# Step 2: Total tokens stored
total_tokens_A = n_chunks_A * chunk_size_A  # each chunk is full size
# Last chunk may be shorter, but typically padded or stored as-is
actual_total_A = (n_chunks_A - 1) * chunk_size_A + min(chunk_size_A, doc_length - (n_chunks_A-1) * stride_A)
total_tokens_B = doc_length  # no overlap = exactly doc length

print(f"\n--- Step 2: Total Tokens Stored ---")
print(f"Strategy A: {n_chunks_A} × {chunk_size_A} = {n_chunks_A * chunk_size_A} tokens "
      f"(actual ≈ {actual_total_A})")
print(f"Strategy B: {doc_length} tokens (no overlap)")

# Step 3: Storage overhead
overhead_A = (actual_total_A - doc_length) / doc_length * 100
overhead_B = 0

print(f"\n--- Step 3: Storage Overhead ---")
print(f"Strategy A: {actual_total_A} vs {doc_length} original = {overhead_A:.1f}% overhead")
print(f"Strategy B: {total_tokens_B} vs {doc_length} original = {overhead_B:.1f}% overhead")

# Embedding storage
d_emb = 768
embed_bytes_A = n_chunks_A * d_emb * 2  # BF16
embed_bytes_B = n_chunks_B * d_emb * 2

print(f"\nEmbedding storage (d={d_emb}, BF16):")
print(f"  Strategy A: {n_chunks_A} embeddings = {fmt_bytes(embed_bytes_A)}")
print(f"  Strategy B: {n_chunks_B} embeddings = {fmt_bytes(embed_bytes_B)}")

# Step 4: Retrieval precision tradeoffs
print(f"\n--- Step 4: Retrieval Tradeoffs ---")
headers = ['Aspect', 'Fixed-256', 'Semantic-180']
rows = [
    ['Chunks', str(n_chunks_A), str(n_chunks_B)],
    ['Avg chunk size', '256 tokens', '~180 tokens'],
    ['Overlap', '64 tokens (25%)', 'None'],
    ['Split quality', 'May split mid-sentence', 'Respects topics'],
    ['Consistency', 'Guaranteed size', 'Variable size'],
    ['Retrieval precision', 'Lower (diluted)', 'Higher (coherent)'],
    ['Context coverage', 'Higher (overlap)', 'Exact (no dup)'],
    ['Implementation', 'Trivial', 'Requires embedding'],
]
print_table(headers, rows, col_width=22)

# Step 5: Content type discussion
print(f"\n--- Step 5: Best Strategy by Content Type ---")
content_types = [
    ('Technical docs', 'Semantic', 'Structure-aware: split at section boundaries'),
    ('Legal contracts', 'Fixed + overlap', 'Dense text; overlap captures cross-clause refs'),
    ('Code', 'Semantic (AST)', 'Split at function/class boundaries'),
    ('Chat logs', 'Fixed', 'Short entries; consistent size works well'),
    ('Scientific papers', 'Semantic', 'Sections are natural chunk boundaries'),
    ('Product catalogs', 'Per-item', 'Each product = one chunk; metadata-rich'),
]
for content, best, reason in content_types:
    print(f"  {content:20s} → {best:18s} ({reason})")

## Exercise 7: NDCG Calculation

Given relevance scores `[3, 0, 2, 1, 0]` for positions 1–5:

1. Compute gains: $\text{gain}_i = 2^{\text{rel}_i} - 1$
2. Compute discounts: $\text{discount}_i = \log_2(i+1)$
3. Compute DCG@5 = Σ gain_i / discount_i
4. Find ideal ranking: sort relevances descending → `[3, 2, 1, 0, 0]`
5. Compute IDCG@5
6. Compute NDCG@5 = DCG@5 / IDCG@5

In [ ]:
# === Exercise 7: Scaffold ===

relevances = [3, 0, 2, 1, 0]
k = 5

# TODO: Step 1 — Gains
# gains = [2^rel - 1 for each rel in relevances]

# TODO: Step 2 — Discounts
# discounts = [log2(i+1) for i in 1..k]  (note: position i is 1-indexed)

# TODO: Step 3 — DCG@5
# dcg = sum(gain_i / discount_i)

# TODO: Step 4 — Ideal ranking
# ideal_rels = sorted(relevances, reverse=True)

# TODO: Step 5 — IDCG@5

# TODO: Step 6 — NDCG@5 = DCG/IDCG
print("Your NDCG calculation:")

In [ ]:
# === Exercise 7: Solution ===

relevances = [3, 0, 2, 1, 0]
k = 5

print("=" * 55)
print("NDCG@5 Computation")
print("=" * 55)

# Step 1: Gains
gains = [2**r - 1 for r in relevances]
print(f"\n--- Step 1: Gains (2^rel - 1) ---")
for i, (r, g) in enumerate(zip(relevances, gains)):
    print(f"  Position {i+1}: rel={r}, gain = 2^{r} - 1 = {g}")

# Step 2: Discounts
discounts = [np.log2(i + 2) for i in range(k)]  # log2(position + 1), position is 1-indexed
print(f"\n--- Step 2: Discounts (log₂(i+1)) ---")
for i, d in enumerate(discounts):
    print(f"  Position {i+1}: discount = log₂({i+2}) = {d:.4f}")

# Step 3: DCG@5
contributions = [g / d for g, d in zip(gains, discounts)]
dcg = sum(contributions)
print(f"\n--- Step 3: DCG@5 ---")
for i, (g, d, c) in enumerate(zip(gains, discounts, contributions)):
    print(f"  Position {i+1}: {g} / {d:.4f} = {c:.4f}")
print(f"  DCG@5 = {dcg:.4f}")

# Step 4: Ideal ranking
ideal_rels = sorted(relevances, reverse=True)
print(f"\n--- Step 4: Ideal Ranking ---")
print(f"  Original:  {relevances}")
print(f"  Ideal:     {ideal_rels}")

# Step 5: IDCG@5
ideal_gains = [2**r - 1 for r in ideal_rels]
ideal_contributions = [g / d for g, d in zip(ideal_gains, discounts)]
idcg = sum(ideal_contributions)
print(f"\n--- Step 5: IDCG@5 ---")
for i, (r, g, d, c) in enumerate(zip(ideal_rels, ideal_gains, discounts, ideal_contributions)):
    print(f"  Position {i+1}: rel={r}, gain={g}, discount={d:.4f}, contribution={c:.4f}")
print(f"  IDCG@5 = {idcg:.4f}")

# Step 6: NDCG@5
ndcg = dcg / idcg
print(f"\n--- Step 6: NDCG@5 ---")
print(f"  NDCG@5 = DCG@5 / IDCG@5 = {dcg:.4f} / {idcg:.4f} = {ndcg:.4f}")
print(f"\n  Interpretation: {ndcg:.0%} of ideal ranking quality")
print(f"  Perfect ranking would give NDCG = 1.0")
print(f"  Lost quality from: rel=2 at position 3 (should be 2), rel=0 at position 2")

# Bonus: NDCG at different k
print(f"\n--- Bonus: NDCG at Different k ---")
for k_val in [1, 2, 3, 4, 5]:
    dcg_k = sum(contributions[:k_val])
    idcg_k = sum(ideal_contributions[:k_val])
    ndcg_k = dcg_k / idcg_k if idcg_k > 0 else 0
    print(f"  NDCG@{k_val} = {dcg_k:.4f} / {idcg_k:.4f} = {ndcg_k:.4f}")

## Exercise 8: RAG Marginalisation

Top-3 retrieved documents with scores `[2.1, 1.8, 0.9]`.
LLM log-probabilities for answer $y$ given each document: `[-1.2, -2.5, -4.1]`.

1. Convert retrieval scores to probabilities with τ = 1.0 (softmax)
2. Convert LLM log-probs to probabilities
3. Compute marginalised P(y|q) = Σ P(d|q) × P(y|q,d)
4. Compute log P(y|q)
5. Compare with top-1 approximation (use only best document)
6. Analyse: when does marginalisation help most?

In [ ]:
# === Exercise 8: Scaffold ===

scores = np.array([2.1, 1.8, 0.9])
llm_log_probs = np.array([-1.2, -2.5, -4.1])
tau = 1.0

# TODO: Step 1 — P(d|q) via softmax with τ=1.0
# retrieval_probs = softmax(scores / tau)

# TODO: Step 2 — P(y|q,d) = exp(log_prob)
# llm_probs = np.exp(llm_log_probs)

# TODO: Step 3 — Marginalise: P(y|q) = Σ P(d|q) × P(y|q,d)
# marginal = sum(retrieval_probs * llm_probs)

# TODO: Step 4 — log P(y|q)
# log_marginal = np.log(marginal)

# TODO: Step 5 — Top-1 comparison
# top1 = llm_probs[0]  # highest retrieval score → doc 0

# TODO: Step 6 — Analysis
print("Your RAG marginalisation:")

In [ ]:
# === Exercise 8: Solution ===

scores = np.array([2.1, 1.8, 0.9])
llm_log_probs = np.array([-1.2, -2.5, -4.1])
tau = 1.0
doc_names = ['D_A', 'D_B', 'D_C']

print("=" * 60)
print("RAG Marginalisation")
print("=" * 60)

# Step 1: Retrieval probabilities
retrieval_probs = softmax(scores, temperature=tau)
print(f"\n--- Step 1: P(d|q) via Softmax (τ={tau}) ---")
print(f"Raw scores: {fmt_vec(scores)}")
for i, (s, p) in enumerate(zip(scores, retrieval_probs)):
    print(f"  P({doc_names[i]}|q) = exp({s})/Σ = {p:.4f}")

# Step 2: LLM probabilities
llm_probs = np.exp(llm_log_probs)
print(f"\n--- Step 2: P(y|q,d) ---")
print(f"Log-probs: {fmt_vec(llm_log_probs)}")
for i, (lp, p) in enumerate(zip(llm_log_probs, llm_probs)):
    print(f"  P(y|q,{doc_names[i]}) = exp({lp}) = {p:.4f}")

# Step 3: Marginalisation
contributions = retrieval_probs * llm_probs
marginal = np.sum(contributions)
print(f"\n--- Step 3: Marginalised P(y|q) ---")
print(f"P(y|q) = Σ P(d|q) × P(y|q,d)")
for i in range(3):
    print(f"  {doc_names[i]}: {retrieval_probs[i]:.4f} × {llm_probs[i]:.4f} = {contributions[i]:.6f}")
print(f"  P(y|q) = {marginal:.6f}")

# Step 4: Log probability
log_marginal = np.log(marginal)
print(f"\n--- Step 4: log P(y|q) ---")
print(f"  log P(y|q) = log({marginal:.6f}) = {log_marginal:.4f}")

# Step 5: Top-1 comparison
top1_prob = llm_probs[0]
top1_log = llm_log_probs[0]
print(f"\n--- Step 5: Top-1 vs Marginalised ---")
print(f"  Top-1 P(y|q)       = P(y|q,D_A) = {top1_prob:.6f} (log: {top1_log:.4f})")
print(f"  Marginalised P(y|q) =             {marginal:.6f} (log: {log_marginal:.4f})")
improvement = (marginal / top1_prob - 1) * 100
print(f"  Improvement: {improvement:.2f}%")
print(f"  Marginalised always ≥ top-1 (by construction)")

# Step 6: Analysis
print(f"\n--- Step 6: When Marginalisation Helps Most ---")

# Case 1: Top doc very confident → marginalisation adds little
scores_case1 = np.array([5.0, 1.0, 0.5])
lp_case1 = np.array([-0.5, -3.0, -5.0])
rp1 = softmax(scores_case1)
m1 = np.sum(rp1 * np.exp(lp_case1))
t1 = np.exp(lp_case1[0])
print(f"\nCase 1: Top doc dominant (score=5.0, log-prob=-0.5)")
print(f"  Top-1: {t1:.6f}, Marginalised: {m1:.6f}, Improvement: {(m1/t1-1)*100:.2f}%")

# Case 2: Multiple docs roughly equal → marginalisation helps a lot
scores_case2 = np.array([1.1, 1.0, 0.9])
lp_case2 = np.array([-1.5, -1.6, -1.7])
rp2 = softmax(scores_case2)
m2 = np.sum(rp2 * np.exp(lp_case2))
t2 = np.exp(lp_case2[0])
print(f"\nCase 2: Docs roughly equal (scores~1.0, log-probs~-1.6)")
print(f"  Top-1: {t2:.6f}, Marginalised: {m2:.6f}, Improvement: {(m2/t2-1)*100:.2f}%")

# Case 3: Second doc has best LLM prob
scores_case3 = np.array([2.0, 1.5, 0.5])
lp_case3 = np.array([-3.0, -0.5, -4.0])
rp3 = softmax(scores_case3)
m3 = np.sum(rp3 * np.exp(lp_case3))
t3 = np.exp(lp_case3[0])
print(f"\nCase 3: Second doc has best P(y|q,d) (log-prob=-0.5)")
print(f"  Top-1: {t3:.6f}, Marginalised: {m3:.6f}, Improvement: {(m3/t3-1)*100:.2f}%")
print(f"  Marginalisation captures signal from non-top-ranked documents!")

print(f"\n--- Summary ---")
print(f"Marginalisation helps most when:")
print(f"  1. Multiple retrieved docs are similarly relevant")
print(f"  2. Non-top-ranked docs have useful information")
print(f"  3. Retrieval scores are uncertain (flat distribution)")
print(f"Marginalisation adds little when top doc dominates overwhelmingly")

print()
print('=' * 60)
print('   ALL EXERCISES COMPLETE')
print('=' * 60)